In [ ]:
# === QuakeMLab sample configuration (auto-added) ===
# By default the workflow runs on the bundled 5-event sample in data/sample/.
# To use your own archive, set QUAKEMLAB_SAMPLE or edit SFILE_PATH/EVENTS_DIR.
import os
SAMPLE = os.environ.get("QUAKEMLAB_SAMPLE") or next(
    (p for p in ("../data/sample", "data/sample") if os.path.isdir(p)), "data/sample")
SFILE_PATH = os.path.join(SAMPLE, "bulletin.out")      # Nordic (SEISAN) S-file
EXCEL_PATH = os.path.join(SAMPLE, "event_list.xlsx")   # optional AFAD list (absent for sample)
EVENTS_DIR = os.path.join(SAMPLE, "events")            # one folder per event, *.mseed (UTC)
DEMO_MODE  = True                                       # M4: run a single lightweight model
def models_to_run(full_list):
    return ["PhaseNetLight"] if DEMO_MODE else full_list
print("SAMPLE =", SAMPLE, "| exists:", os.path.isdir(SAMPLE))


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import json
import numpy as np
import pandas as pd
from obspy import read, UTCDateTime
import warnings
import seisbench.models as sbm

warnings.filterwarnings("ignore")

import os
import warnings
import pandas as pd
from obspy import read
import seisbench.models as sbm
from tqdm import tqdm

# OpenMP çakışması için (Windows)
import os
import seisbench.models as sbm

json_file = "sfile_parsed.json"
mseed_root = EVENTS_DIR
mseed_root_demo = EVENTS_DIR
out_dir = "seisbench_ready"

def extract_event_id_from_filename(filename):
    """
    Dosya isminden event_id'yi alır.
    Örnek: '286369_20150108_061732_ML_30.mseed' -> 286369
    """
    base = os.path.basename(filename)
    event_id_str = base.split('_')[0]
    return int(event_id_str)

In [ ]:
class SeisBenchPreparer:
    def __init__(self, json_file, mseed_root, out_dir, target_sr=100.0):
        self.json_file = json_file
        self.mseed_root = mseed_root
        self.out_dir = out_dir
        self.target_sr = target_sr
        
        os.makedirs(out_dir, exist_ok=True)
        self.out_csv = os.path.join(out_dir, "seisbench_ready.csv")        
        # JSON yükle
        with open(json_file, "r") as f:
            self.events = json.load(f)
        self.model=None
    
    def preprocess_stream(self, st):
        st = st.copy()
        try:
            st.merge(fill_value=0)
        except:
            pass
        
        for tr in st:
            tr.detrend("demean")
            tr.detrend("linear")
            try:
                tr.filter("bandpass", freqmin=1, freqmax=45)
            except:
                pass
            tr.taper(max_percentage=0.001, type="cosine", max_length=2)
            
            if abs(tr.stats.sampling_rate - self.target_sr) > 1e-2:
                try:
                    tr.resample(self.target_sr)
                except:
                    pass
        return st
    
    def process_event(self, event):
        event_folder = event["event_folder_name"]
        folder_path = os.path.join(self.mseed_root, event_folder)
        if not os.path.exists(folder_path):
            print(f"❌ {event_folder} yok")
            return []
        
        mseed_files = [os.path.join(folder_path,f) for f in os.listdir(folder_path) if f.lower().endswith(".mseed")]
        if len(mseed_files)==0:
            print(f"❌ {event_folder} içinde mseed yok")
            return []
        
        rows = []
        for fpath in mseed_files:
            fname = os.path.basename(fpath)
            try:
                st = read(fpath)
            except:
                continue
            st = self.preprocess_stream(st)
            
            # PhaseNet model çağrısı tamamen kaldırıldı
            try:
                picks = self.model.classify(st)
            except:
                picks = []
            
            for tr in st:
                # waveform .npy kaydet
                npy_file = os.path.join(self.out_dir, f"{fname}_{tr.id}.npy")
                if not os.path.exists(npy_file):
                    np.save(npy_file, tr.data)
                # manuel pickler varsa ekle
                # manual_picks = [p for p in event.get("picks", []) if p["station"]==tr.stats.station]
                
                # PhaseNet pick kodları kaldırıldı
                # phasenet_picks = []
                # if hasattr(picks, "picks"):
                #     for p in picks.picks:
                #         if tr.id.startswith(p.trace_id.split('.')[1]):
                #             phasenet_picks.append({
                #                 "phase": p.phase,
                #                 "time": str(p.peak_time)
                #             })
                
                # CSV satırı oluştur
                row = {
                    "event_id": event["event_folder_name"],
                    "station": tr.stats.station,
                    "channel": tr.stats.channel,
                    "network": tr.stats.network,
                    "start_time": str(tr.stats.starttime),
                    "end_time": str(tr.stats.endtime),
                    "sampling_rate": tr.stats.sampling_rate,
                    # "manual_picks": json.dumps(manual_picks),
                    # "phasenet_picks": json.dumps(phasenet_picks),
                    "waveform_file": npy_file
                }
                rows.append(row)
        return rows
    
    def run(self):
        all_rows = []
        for event in self.events:
            rows = self.process_event(event)
            all_rows.extend(rows)
        df = pd.DataFrame(all_rows)
        df.to_csv(self.out_csv, index=False)
        print(f"✅ SeisBench uyumlu CSV oluşturuldu: {self.out_csv}")

In [ ]:
prep = SeisBenchPreparer(json_file, mseed_root, out_dir)
prep.run()

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from obspy import read
from tqdm import tqdm

class SeisBenchPreparer:
    def __init__(self, json_file, mseed_root, out_dir, target_sr=100.0):
        self.json_file = json_file
        self.mseed_root = mseed_root
        self.out_dir = out_dir
        self.target_sr = target_sr
        
        # Ana çıktı klasörünü oluştur
        os.makedirs(out_dir, exist_ok=True)
        self.out_csv = os.path.join(out_dir, "seisbench_ready.csv")        
        
        # JSON yükle
        with open(json_file, "r", encoding='utf-8') as f:
            self.events = json.load(f)
    
    def preprocess_stream(self, st):
        """Sinyal işleme: Merge, Detrend, Filter, Taper, Resample"""
        st = st.copy()
        try:
            st.merge(fill_value=0)
        except Exception:
            pass
        
        for tr in st:
            try:
                tr.detrend("demean")
                tr.detrend("linear")
                tr.filter("bandpass", freqmin=1.0, freqmax=45.0)
                tr.taper(max_percentage=0.001, type="cosine", max_length=2)
                
                if abs(tr.stats.sampling_rate - self.target_sr) > 1e-2:
                    tr.resample(self.target_sr)
            except Exception:
                pass
                
        return st
    
    def process_event(self, event):
        # Kaynak klasör adı ve Event ID
        event_folder_name = event.get("event_folder_name")
        event_id = str(event.get("event_id"))
        
        if not event_folder_name or not event_id:
            return []

        # Kaynak mseed klasörünü bul
        source_path = os.path.join(self.mseed_root, event_folder_name)
        if not os.path.exists(source_path):
            return []
        
        # Hedef alt klasörü oluştur: sisbench_ready/286369/
        target_event_dir = os.path.join(self.out_dir, event_id)
        os.makedirs(target_event_dir, exist_ok=True)
        
        # Mseed dosyalarını listele
        mseed_files = [os.path.join(source_path, f) for f in os.listdir(source_path) 
                       if f.lower().endswith((".mseed", ".miniseed"))]
        
        if not mseed_files:
            return []
        
        rows = []
        for fpath in mseed_files:
            fname = os.path.basename(fpath)
            try:
                st = read(fpath)
            except Exception:
                continue
            
            # Preprocessing
            st = self.preprocess_stream(st)
            
            for tr in st:
                # --- İSİMLENDİRME DÜZELTMESİ ---
                # İstenen Format: 286369_..._30.mseed_KO.BALB..HHE.npy
                # Mantık: {DosyaAdı}_{TraceID}.npy
                unique_filename = f"{fname}_{tr.id}.npy"
                
                # Tam Kayıt Yolu
                full_save_path = os.path.join(target_event_dir, unique_filename)
                
                # Göreli Yol (CSV için)
                relative_path = os.path.join(event_id, unique_filename)
                
                # --- OPTİMİZASYON: Dosya Varsa Yazma ---
                if not os.path.exists(full_save_path):
                    np.save(full_save_path, tr.data)
                
                # CSV Metadatasını Hazırla (Dosya olsa bile CSV'ye eklemeliyiz)
                row = {
                    "event_id": event_id,
                    "event_folder": event_folder_name,
                    "network": tr.stats.network,
                    "station": tr.stats.station,
                    "location": tr.stats.location,
                    "channel": tr.stats.channel,
                    "start_time": str(tr.stats.starttime),
                    "end_time": str(tr.stats.endtime),
                    "sampling_rate": tr.stats.sampling_rate,
                    "npts": tr.stats.npts,
                    "waveform_file": relative_path, 
                    "full_path": full_save_path
                }
                rows.append(row)
                
        return rows
    
    def run(self):
        print(f"🚀 Veri Hazırlığı Başlıyor...")
        print(f"📂 Hedef: {self.out_dir}")
        
        all_rows = []
        
        # TQDM ile ilerleme
        for event in tqdm(self.events, desc="Eventler Taranıyor"):
            rows = self.process_event(event)
            all_rows.extend(rows)
            
        if all_rows:
            df = pd.DataFrame(all_rows)
            df.to_csv(self.out_csv, index=False)
            print(f"✅ İşlem Tamamlandı! {len(all_rows)} trace indekslendi.")
            print(f"📄 İndeks dosyası: {self.out_csv}")
        else:
            print("❌ Hiçbir veri işlenemedi.")

# --- KULLANIM ---
if __name__ == "__main__":
    JSON_FILE = "sfile_parsed.json"
    MSEED_ROOT = EVENTS_DIR 
    OUT_DIR = "seisbench_ready"
    
    preparer = SeisBenchPreparer(JSON_FILE, MSEED_ROOT, OUT_DIR)
    preparer.run()